<a href="https://colab.research.google.com/github/dbonacorsi/AML-Basic/blob/main/AMLBas_2_DataPrep_ExploreData_part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Explore your data with descriptive statistics

***The time spent in understanding your data (BEFORE APPLYING ANY ML TECHNIQUES) is time well spent***.

In this notebook we start exploring **EDA** (**exploratory data analysis**): the process of analyzing and understanding your dataset before building models.

Suggestions and guidelines may differ, but here is what we use as a checklist:

1. Look at the raw data
1. Dimensions
1. Data types
1. Descriptive statistics (incl. missing values)
1. Distribution of instances across classes
1. Correlations
1. Skews

## 1. Raw data

In [1]:
import pandas as pd

url = 'https://raw.githubusercontent.com/dbonacorsi/AML-Basic/refs/heads/main/datasets/pima-indians-diabetes.data.csv'

names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
data = pd.read_csv(url, names=names)
data

,preg,plas,pres,skin,test,mass,pedi,age,class
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


More details can be found e.g. at https://www.kaggle.com/uciml/pima-indians-diabetes-database/data, and are summarized below:

    Pregnancies = Number of times pregnant
    Glucose = Plasma glucose concentration a 2 hours in an oral glucose tolerance test numeric
    BloodPressure = Diastolic blood pressure (mm Hg)
    SkinThickness = Triceps skin fold thickness (mm)
    Insulin = 2-Hour serum insulin (mu U/ml)
    BMI = Body mass index (weight in kg/(height in m)^2)
    DiabetesPedigreeFunction = Diabetes pedigree function
    Age = Age (years)
    Outcome = Class variable (0 or 1)

This is the mapping to actual code short names, in the same order:

    'preg'
    'plas'
    'pres'
    'skin'
    'test'
    'mass'
    'pedi'
    'age'
    'class'

## 2. Dimensions

In [3]:
shape = data.shape
shape

(768, 9)

## 3. Data types

In [4]:
types = data.dtypes
print(types)

preg       int64
plas       int64
pres       int64
skin       int64
test       int64
mass     float64
pedi     float64
age        int64
class      int64
dtype: object


## 4. Descriptive statistics

Help on pandas set_option [here](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.set_option.html).


In [26]:
from pandas import set_option

set_option('display.width', 200)
set_option('display.max_rows', 500)
set_option('display.max_columns', 500)
set_option('display.precision', 3)

In [27]:
description = data.describe()
print(description)

          preg     plas     pres     skin     test     mass     pedi      age    class
count  768.000  768.000  768.000  768.000  768.000  768.000  768.000  768.000  768.000
mean     3.845  120.895   69.105   20.536   79.799   31.993    0.472   33.241    0.349
std      3.370   31.973   19.356   15.952  115.244    7.884    0.331   11.760    0.477
min      0.000    0.000    0.000    0.000    0.000    0.000    0.078   21.000    0.000
25%      1.000   99.000   62.000    0.000    0.000   27.300    0.244   24.000    0.000
50%      3.000  117.000   72.000   23.000   30.500   32.000    0.372   29.000    0.000
75%      6.000  140.250   80.000   32.000  127.250   36.600    0.626   41.000    1.000
max     17.000  199.000  122.000   99.000  846.000   67.100    2.420   81.000    1.000


What can we learn from this simple overview of our data?
1.   Are there missing entries anywhere? Anything suspicious? (*)
2.   Are the values of the features on the same scales? (**)



(*) e.g. can you explain the fact that the describe() output shows min = 0 for features that cannot be 0 physiologically?


(**) not at all.. more on this in one of next notebooks!

In [14]:
# Features where 0 is physiologically impossible: pandas reads them as 0, not as NaN
cols = ['plas', 'pres', 'skin', 'test', 'mass']

# Count the 0 placeholders per column
print("Zeros per column (likely missing values):")
print((data[cols] == 0).sum())

# Count real NaN values (expected to be 0 everywhere, since the file has no blank fields)
print("\nNaN values per column:")
print(data.isnull().sum())

Zeros per column (likely missing values):
plas      5
pres     35
skin    227
test    374
mass     11
dtype: int64

NaN values per column:
preg     0
plas     0
pres     0
skin     0
test     0
mass     0
pedi     0
age      0
class    0
dtype: int64


Let's discuss this.

**Should I remove the features?**. No, at least not yet.

Why? Dropping a whole feature throws away the other 767 valid values in it, and most of the zeros are only a few percent of the rows in three of the five columns. The usual approach is to treat the zeros as missing, then decide how to fill them.

Your options, roughly in order of preference:

*   *Replace 0 with NaN, then impute (recommended)*. Fill each gap with the column median, which is less sensitive to outliers than the mean. Do this inside a pipeline, fitted on the training split only, so the test set doesn’t leak into the fill values.

*   *Drop the rows with missing values*. Simple, but test (374 zeros) would remove about half the dataset, so it’s only reasonable for small losses.

*   *Drop the feature*. This is defensible for test (about half its values are missing) and possibly skin (227 of 768). Decide it on evidence: compare model accuracy with and without it, rather than by eye.

*   *Add a “was missing” indicator column* (for example test_missing), so the model can learn from the fact that the value was absent. Consider treat this as a one-hot-encoded info?

At the moment we are just exploring and familiarizing, exploiting this (very educational) dataset. But it this would have been a real-world dataset and a real ML project, this aspect would have been a point to seriously address!


## 5. Class distribution
(only for classification problems)

Just counting..

In [7]:
class_counts = data.groupby('class').size()
print(class_counts)

class
0    500
1    268
dtype: int64


Or checking proportions..

In [17]:
class_proportions = data['class'].value_counts(normalize=True)
print(class_proportions.round(3))

class
0    0.651
1    0.349
Name: proportion, dtype: float64


The split is about 65/35, which is a mild imbalance: it is worth mentioning that - in this mild case already - accuracy alone might be misleading.. be carefull (if this would be a real project!)

## 6. Correlations between attributes

In [20]:
set_option('display.width', 100)
#set_option('precision', 3)

correlations = data.corr(method='pearson')
correlations

,preg,plas,pres,skin,test,mass,pedi,age,class
preg,1.000000,0.129459,0.141282,-0.081672,-0.073535,0.017683,-0.033523,0.544341,0.221898
plas,0.129459,1.000000,0.152590,0.057328,0.331357,0.221071,0.137337,0.263514,0.466581
pres,0.141282,0.152590,1.000000,0.207371,0.088933,0.281805,0.041265,0.239528,0.065068
skin,-0.081672,0.057328,0.207371,1.000000,0.436783,0.392573,0.183928,-0.113970,0.074752
test,-0.073535,0.331357,0.088933,0.436783,1.000000,0.197859,0.185071,-0.042163,0.130548
mass,0.017683,0.221071,0.281805,0.392573,0.197859,1.000000,0.140647,0.036242,0.292695
pedi,-0.033523,0.137337,0.041265,0.183928,0.185071,0.140647,1.000000,0.033561,0.173844
age,0.544341,0.263514,0.239528,-0.113970,-0.042163,0.036242,0.033561,1.000000,0.238356
class,0.221898,0.466581,0.065068,0.074752,0.130548,0.292695,0.173844,0.238356,1.000000


Show which features correlate most with the target (NOTE: Pearson correlation only captures linear relationships, though..)

In [18]:
correlations['class'].sort_values(ascending=False)

,class
class,1.000000
plas,0.466581
mass,0.292695
age,0.238356
preg,0.221898
pedi,0.173844
test,0.130548
skin,0.074752
pres,0.065068


## 7. Skew of Univariate Distributions

In [9]:
skew = data.skew()
print(skew)

preg     0.901674
plas     0.173754
pres    -1.843608
skin     0.109372
test     2.272251
mass    -0.428982
pedi     1.919911
age      1.129597
class    0.635017
dtype: float64


Rule of thumb: e.g. `|skew| > 1` means strongly skewed.


## Summary

What we did:

* we familiarized with some quick ways to have a glance at my dataset and try to describe and summarize it statistically
* we implemented these concepts by exploiting Pandas' functionalities: easy, basically one command each.
* remember to look carefully at the numbers, ask yourself questions as of whether things are as you see them, and note down evertything that comes to mind as it might well be useful at a later stage!

## What's next

We did what we did with numbers. ***Isn't it frustrating?*** Time to make it with visualizations!